<a href="https://colab.research.google.com/github/Boni1995/DSE_thesis/blob/main/Airbnb_Translation_(cleaned).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

In [ ]:
upload = files.upload()

In [ ]:
filename = next(iter(upload))

In [ ]:
df_speeches = pd.read_excel(filename)

In [ ]:
# Create model to translate from spanish to english, so it can be implemented correctly to the final model

language_model_name = "Helsinki-NLP/opus-mt-es-en"
language_tokenizer = MarianTokenizer.from_pretrained(language_model_name)
language_model = MarianMTModel.from_pretrained(language_model_name)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
# Defining GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

language_model.to(device)
language_model.eval()

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(65001, 512, padding_idx=65000)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(65001, 512, padding_idx=65000)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [ ]:
def translate(texts):
    """
    Translates a list of texts from a source language to a target language using a transformer-based translation model.

    Parameters:
        texts (list of str): List of input text strings to be translated.

    Returns:
        list of str: List of translated text strings corresponding to each input text, in the same order.
    """

    inputs = language_tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

    # Move tokenized tensors to GPU
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        with amp.autocast():
            translated = language_model.generate(**inputs, max_length=128, num_beams=1)

    outputs = [language_tokenizer.decode(t, skip_special_tokens=True) for t in translated]

    return outputs